# Simply Generate — Qwen3-MoE

Quick prompt sandbox for the qwen3-moe-port branch. The three Qwen-specific patches (Blackwell `_grouped_mm` fallback, fp16→bf16 cast on `lm_head`, `enable_thinking=False`) live in `cv_utils.py`, so this notebook is a thin wrapper.

Use it to sanity-check a prompt before running the full extraction pipeline.

In [ ]:
import torch
from cv_utils import load_model

model = load_model()
print('num layers:', len(model.model.layers))

## Prompt under test

Edit the `concept` and `topic` fields to validate any of the seven emotions before kicking off `extract_concepts.py`. Watch for over-avoidance (model describes the event with no character-level emotional reaction) — that's the failure mode this cell is meant to catch.

In [ ]:
concept = 'surprise'
topic = 'A neighbor starts a renovation project.'

prompt = f"""Write a story based on the following premise.

Topic: {topic}

The story should follow a character who is feeling {concept}.

IMPORTANT: You must NEVER use the word '{concept}' or any direct synonyms of it in the stories. Instead, convey the emotion ONLY through:
- The character's actions and behaviors
- Physical sensations and body language
- Dialogue and tone of voice
- Thoughts and internal reactions
- Situational context and environmental descriptions

The emotion should be clearly conveyed to the reader through these indirect means, but never explicitly named.

Story:
"""
print(prompt)

In [ ]:
input_text = model.tokenizer.apply_chat_template(
    [{'role': 'user', 'content': prompt}],
    tokenize=False, add_generation_prompt=True,
    enable_thinking=False,
)
prompt_len = model.tokenizer(input_text, return_tensors='pt').input_ids.shape[1]

with model.generate(input_text, max_new_tokens=600, do_sample=True,
                    temperature=0.8, top_p=0.9,
                    pad_token_id=model.tokenizer.eos_token_id):
    out_ids = model.generator.output.save()

completion = model.tokenizer.decode(out_ids[0, prompt_len:].cpu(), skip_special_tokens=True)
print(completion)